# 🏆 Gemma 4 Good AI for Humanity - Final Submission

**Project:** Gemma 4 - Good AI for Humanity, Not Just Performance  
**Notebook:** Final submission combining best models and prompt strategies  
**Author:** Data Science Team  
**Last Updated:** 2024-03-01  
**Version:** Final v1.0

## 📋 Table of Contents
1. [Introduction & Objective](#introduction)
2. [Configuration](#configuration)
3. [Data Loading](#data-loading)
4. [Feature Pipeline](#feature-pipeline)
5. [Model Ensemble](#model-ensemble)
6. [Optimized Prompt Strategy](#prompt-strategy)
7. [Prediction on Test Set](#prediction)
8. [Output Generation](#output)
9. [Model Card & Documentation](#model-card)

---
## 1. Introduction & Objective <a name="introduction"></a>

### Goal
Predict high-quality, humanity-focused AI responses that prioritize **human welfare, empathy, and social good** alongside strong performance.

### Approach
This final submission combines:
- **Structural features** from `baseline_model.ipynb`
- **TF-IDF semantic features** from `tfidf_experiment.ipynb`
- **Optimized prompt engineering** from `gemma_prompt_test.ipynb`
- **Domain insights** from `eda_environment.ipynb` and `eda_health.ipynb`

### Scoring Formula
```
composite_score = humanity_score * 0.35 + performance_score * 0.25 + helpfulness_score * 0.40
```

In [ ]:
# ============================================
# Cell 1: Imports
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries loaded successfully!")

---
## 2. Configuration <a name="configuration"></a>

In [ ]:
# ============================================
# Cell 2: Configuration
# ============================================
CONFIG = {
    # Data paths
    'train_path': '../data/raw/train.csv',
    'test_path': '../data/raw/test.csv',
    'labels_path': '../data/raw/labels.csv',
    'output_dir': '../data/processed/',
    
    # TF-IDF parameters
    'tfidf_max_features': 1000,
    'tfidf_ngram_range': (1, 3),
    'svd_components': 100,
    
    # Scoring weights
    'humanity_weight': 0.35,
    'performance_weight': 0.25,
    'helpfulness_weight': 0.40,
    
    # Model parameters
    'random_state': 42,
    'cv_folds': 5
}

# Optimal prompt strategy (from gemma_prompt_test.ipynb)
OPTIMAL_STRATEGY = {
    'system_prompt': (
        'You are a humanitarian AI advisor with expertise in ethical technology, '
        'public health, and international development. You approach every question '
        with deep empathy, evidence-based reasoning, and a commitment to serving '
        'the most vulnerable communities. Your responses are structured, actionable, '
        'and always consider diverse perspectives and potential unintended consequences.'
    ),
    'response_framework': [
        '1. Acknowledge the human dimension of the question',
        '2. Provide clear, evidence-based analysis',
        '3. Consider impact on vulnerable populations',
        '4. Offer actionable, practical recommendations',
        '5. Highlight equity and inclusion considerations'
    ]
}

print("Configuration loaded!")
print(f"  Scoring weights: H={CONFIG['humanity_weight']}, P={CONFIG['performance_weight']}, Help={CONFIG['helpfulness_weight']}")
print(f"  TF-IDF features: {CONFIG['tfidf_max_features']}, SVD components: {CONFIG['svd_components']}")

---
## 3. Data Loading <a name="data-loading"></a>

In [ ]:
# ============================================
# Cell 3: Load Data
# ============================================
train_df = pd.read_csv(CONFIG['train_path'])
test_df = pd.read_csv(CONFIG['test_path'])
labels_df = pd.read_csv(CONFIG['labels_path'])

# Merge train with labels
train_merged = train_df.merge(
    labels_df[['id', 'safety_score', 'fairness_score', 'clarity_score']],
    on='id', how='left'
)

# Compute composite score
train_merged['composite_score'] = (
    train_merged['humanity_score'] * CONFIG['humanity_weight'] +
    train_merged['performance_score'] * CONFIG['performance_weight'] +
    train_merged['helpfulness_score'] * CONFIG['helpfulness_weight']
)

print(f"Training set:   {train_merged.shape}")
print(f"Test set:       {test_df.shape}")
print(f"\nComposite score stats:")
print(f"  Mean: {train_merged['composite_score'].mean():.3f}")
print(f"  Std:  {train_merged['composite_score'].std():.3f}")
print(f"  Range: [{train_merged['composite_score'].min():.2f}, {train_merged['composite_score'].max():.2f}]")

---
## 4. Feature Pipeline <a name="feature-pipeline"></a>

In [ ]:
# ============================================
# Cell 4: Structural Feature Engineering
# ============================================
def create_structural_features(df):
    """Create structural and statistical features from text data."""
    feat = df.copy()
    
    # Length features
    feat['prompt_length'] = feat['prompt'].str.len()
    feat['response_length'] = feat['response'].str.len()
    feat['prompt_word_count'] = feat['prompt'].str.split().str.len()
    feat['response_word_count'] = feat['response'].str.split().str.len()
    feat['length_ratio'] = feat['response_word_count'] / feat['prompt_word_count'].replace(0, 1)
    
    # Structural features
    feat['sentence_count'] = feat['response'].apply(lambda x: len(re.split(r'[.!?]+', str(x))))
    feat['avg_sentence_length'] = feat['response_word_count'] / feat['sentence_count'].replace(0, 1)
    feat['has_list'] = feat['response'].apply(lambda x: int(bool(re.search(r'(\d\)|-\s|\*)', str(x)))))
    feat['has_number'] = feat['response'].apply(lambda x: int(bool(re.search(r'\d', str(x)))))
    feat['paragraph_count'] = feat['response'].apply(lambda x: max(1, str(x).count('\n\n') + 1))
    
    # Lexical features
    feat['unique_word_count'] = feat['response'].apply(lambda x: len(set(str(x).lower().split())))
    feat['lexical_diversity'] = feat['unique_word_count'] / feat['response_word_count'].replace(0, 1)
    feat['avg_word_length'] = feat['response'].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if pd.notna(x) else 0
    )
    
    # Punctuation
    feat['question_marks'] = feat['response'].str.count('\?')
    feat['exclamation_marks'] = feat['response'].str.count('!')
    feat['comma_count'] = feat['response'].str.count(',')
    
    # Category encoding
    le_cat = LabelEncoder()
    le_diff = LabelEncoder()
    le_lang = LabelEncoder()
    
    feat['category_encoded'] = le_cat.fit_transform(feat['category'])
    feat['difficulty_encoded'] = le_diff.fit_transform(feat['difficulty'])
    feat['language_encoded'] = le_lang.fit_transform(feat['language'])
    
    return feat, le_cat, le_diff, le_lang

train_feat, le_cat, le_diff, le_lang = create_structural_features(train_merged)
print(f"Structural features created: {train_feat.shape[1]} columns")

In [ ]:
# ============================================
# Cell 5: TF-IDF Feature Engineering
# ============================================
custom_stop_words = list(ENGLISH_STOP_WORDS) + [
    'also', 'would', 'could', 'should', 'however', 'therefore',
    'furthermore', 'addition', 'example', 'include', 'including',
    'make', 'way', 'many', 'much', 'well', 'like', 'need'
]

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_feat['clean_response'] = train_feat['response'].apply(preprocess_text)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=CONFIG['tfidf_max_features'],
    ngram_range=CONFIG['tfidf_ngram_range'],
    stop_words=custom_stop_words,
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(train_feat['clean_response'])

# SVD Dimensionality Reduction
svd = TruncatedSVD(n_components=CONFIG['svd_components'], random_state=CONFIG['random_state'])
tfidf_reduced = svd.fit_transform(tfidf_matrix)

print(f"TF-IDF matrix: {tfidf_matrix.shape}")
print(f"SVD reduced:   {tfidf_reduced.shape}")
print(f"Variance explained: {svd.explained_variance_ratio_.sum():.3f}")

In [ ]:
# ============================================
# Cell 6: Combine All Features
# ============================================
STRUCTURAL_COLS = [
    'prompt_length', 'response_length', 'prompt_word_count', 'response_word_count',
    'length_ratio', 'sentence_count', 'avg_sentence_length', 'has_list', 'has_number',
    'paragraph_count', 'unique_word_count', 'lexical_diversity', 'avg_word_length',
    'question_marks', 'exclamation_marks', 'comma_count',
    'category_encoded', 'difficulty_encoded', 'language_encoded'
]

ANNOTATION_COLS = ['safety_score', 'fairness_score', 'clarity_score']
available_annot = [c for c in ANNOTATION_COLS if c in train_feat.columns]

X_structural = train_feat[STRUCTURAL_COLS + available_annot].values
X_combined = np.hstack([X_structural, tfidf_reduced])
y = train_feat['composite_score'].values

print(f"Combined feature matrix: {X_combined.shape}")
print(f"  Structural features: {X_structural.shape[1]}")
print(f"  TF-IDF features: {tfidf_reduced.shape[1]}")

---
## 5. Model Ensemble <a name="model-ensemble"></a>

In [ ]:
# ============================================
# Cell 7: Train Stacking Ensemble
# ============================================
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Define base models
base_models = [
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=CONFIG['random_state'], n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=CONFIG['random_state'])),
    ('svr', SVR(kernel='rbf', C=10, epsilon=0.1))
]

# Stacking ensemble
stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=0.5),
    cv=CONFIG['cv_folds'],
    n_jobs=-1
)

print("Training stacking ensemble...")
stacking.fit(X_scaled, y)
print("Training complete!")

In [ ]:
# ============================================
# Cell 8: Cross-Validation Scores
# ============================================
cv_scores = cross_val_score(
    stacking, X_scaled, y,
    cv=CONFIG['cv_folds'],
    scoring='neg_root_mean_squared_error'
)

print(f"Cross-Validation Results ({CONFIG['cv_folds']}-fold):")
print(f"  RMSE: {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"  Individual folds: {[round(-s, 4) for s in cv_scores]}")

# Training predictions
y_train_pred = stacking.predict(X_scaled)
train_rmse = np.sqrt(mean_squared_error(y, y_train_pred))
train_mae = mean_absolute_error(y, y_train_pred)
train_r2 = r2_score(y, y_train_pred)

print(f"\nTraining Set Metrics:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R2:   {train_r2:.4f}")

In [ ]:
# ============================================
# Cell 9: Actual vs Predicted Visualization
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y, y_train_pred, alpha=0.7, color='#3498db', edgecolors='gray', s=80)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('Actual Composite Score')
axes[0].set_ylabel('Predicted Composite Score')
axes[0].set_title('Actual vs Predicted (Training)', fontsize=12, fontweight='bold')
axes[0].legend(loc='best')

# Residual distribution
residuals = y - y_train_pred
axes[1].hist(residuals, bins=15, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}final_model_performance.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Optimized Prompt Strategy <a name="prompt-strategy"></a>

In [ ]:
# ============================================
# Cell 10: Apply Optimal Prompt Strategy to Test Set
# ============================================
def apply_optimal_prompt(prompt_text):
    """Apply the winning prompt engineering strategy."""
    return f"""{prompt_text}

Please structure your response following this framework:
- Begin with the human dimension and real-world impact
- Provide clear, evidence-based analysis
- Consider effects on vulnerable and marginalized populations
- Offer specific, actionable recommendations
- Highlight equity, inclusion, and sustainability considerations

Aim to be empathetic, practical, and thorough."""

# Apply to test prompts
test_df['enhanced_prompt'] = test_df['prompt'].apply(apply_optimal_prompt)

print(f"Enhanced {len(test_df)} test prompts with optimal strategy.")
print(f"\nExample enhanced prompt:\n{test_df['enhanced_prompt'].iloc[0][:300]}...")

---
## 7. Prediction on Test Set <a name="prediction"></a>

In [ ]:
# ============================================
# Cell 11: Generate Test Features
# ============================================
# Apply same feature engineering to test set
test_feat, _, _, _ = create_structural_features(test_df)

# For test set with missing responses, use prompt-based features only
# When responses are generated, re-run this pipeline
print(f"Test features prepared: {test_feat.shape}")
print(f"\nNote: Full prediction requires generated responses for TF-IDF features.")
print(f"Use the enhanced_prompt column with Gemma API to generate responses,")
print(f"then re-run the feature pipeline to get final predictions.")

In [ ]:
# ============================================
# Cell 12: Prediction Pipeline (with generated responses)
# ============================================
def predict_scores(df, model, scaler, tfidf, svd, le_cat, le_diff, le_lang):
    """Full prediction pipeline for new data."""
    
    # Structural features
    df['prompt_length'] = df['prompt'].str.len()
    df['response_length'] = df['response'].str.len()
    df['prompt_word_count'] = df['prompt'].str.split().str.len()
    df['response_word_count'] = df['response'].str.split().str.len()
    df['length_ratio'] = df['response_word_count'] / df['prompt_word_count'].replace(0, 1)
    df['sentence_count'] = df['response'].apply(lambda x: len(re.split(r'[.!?]+', str(x))))
    df['avg_sentence_length'] = df['response_word_count'] / df['sentence_count'].replace(0, 1)
    df['has_list'] = df['response'].apply(lambda x: int(bool(re.search(r'(\d\)|-\s|\*)', str(x)))))
    df['has_number'] = df['response'].apply(lambda x: int(bool(re.search(r'\d', str(x)))))
    df['paragraph_count'] = df['response'].apply(lambda x: max(1, str(x).count('\n\n') + 1))
    df['unique_word_count'] = df['response'].apply(lambda x: len(set(str(x).lower().split())))
    df['lexical_diversity'] = df['unique_word_count'] / df['response_word_count'].replace(0, 1)
    df['avg_word_length'] = df['response'].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if pd.notna(x) else 0
    )
    df['question_marks'] = df['response'].str.count('\?')
    df['exclamation_marks'] = df['response'].str.count('!')
    df['comma_count'] = df['response'].str.count(',')
    
    # Encode categoricals (handle unseen labels)
    df['category_encoded'] = df['category'].map(
        dict(zip(le_cat.classes_, le_cat.transform(le_cat.classes_)))
    ).fillna(-1)
    df['difficulty_encoded'] = df['difficulty'].map(
        dict(zip(le_diff.classes_, le_diff.transform(le_diff.classes_)))
    ).fillna(-1)
    df['language_encoded'] = df['language'].map(
        dict(zip(le_lang.classes_, le_lang.transform(le_lang.classes_)))
    ).fillna(-1)
    
    # TF-IDF features
    df['clean_response'] = df['response'].apply(preprocess_text)
    tfidf_test = tfidf.transform(df['clean_response'])
    svd_test = svd.transform(tfidf_test)
    
    # Combine
    X_struct = df[STRUCTURAL_COLS].values
    X_all = np.hstack([X_struct, svd_test])
    X_scaled = scaler.transform(X_all)
    
    # Predict
    predictions = model.predict(X_scaled)
    return np.clip(predictions, 1.0, 5.0)

print("Prediction pipeline defined!")

---
## 8. Output Generation <a name="output"></a>

In [ ]:
# ============================================
# Cell 13: Generate Submission File
# ============================================
# Create submission template
submission = test_df[['id', 'prompt', 'category', 'difficulty']].copy()
submission['system_prompt'] = OPTIMAL_STRATEGY['system_prompt']
submission['enhanced_prompt'] = test_df['enhanced_prompt']
submission['response'] = ''  # Fill after model inference
submission['predicted_composite_score'] = np.nan  # Fill after inference

# Save submission template
submission.to_csv(f"{CONFIG['output_dir']}submission_template.csv", index=False)

print(f"Submission template saved: {submission.shape}")
print(submission.columns.tolist())

In [ ]:
# ============================================
# Cell 14: Save Model Artifacts
# ============================================
import pickle

artifacts = {
    'model': stacking,
    'scaler': scaler,
    'tfidf': tfidf,
    'svd': svd,
    'label_encoders': {
        'category': le_cat,
        'difficulty': le_diff,
        'language': le_lang
    },
    'config': CONFIG,
    'prompt_strategy': OPTIMAL_STRATEGY
}

with open(f"{CONFIG['output_dir']}final_model_artifacts.pkl", 'wb') as f:
    pickle.dump(artifacts, f)

print("Model artifacts saved!")

---
## 9. Model Card & Documentation <a name="model-card"></a>

In [ ]:
# ============================================
# Cell 15: Model Card
# ============================================
model_card = {
    'model_name': 'Gemma 4 - Good AI for Humanity',
    'version': '1.0',
    'description': (
        'A stacked ensemble model for predicting AI response quality with emphasis '
        'on humanity, helpfulness, and social good metrics.'
    ),
    'architecture': {
        'type': 'StackingRegressor',
        'base_models': ['Ridge', 'RandomForest', 'GradientBoosting', 'SVR'],
        'meta_model': 'Ridge',
        'features': {
            'structural': len(STRUCTURAL_COLS),
            'tfidf_svd': CONFIG['svd_components'],
            'total': len(STRUCTURAL_COLS) + CONFIG['svd_components']
        }
    },
    'training': {
        'cv_rmse': round(-cv_scores.mean(), 4),
        'cv_std': round(cv_scores.std(), 4),
        'train_rmse': round(train_rmse, 4),
        'train_r2': round(train_r2, 4),
        'n_samples': len(y)
    },
    'scoring': {
        'composite_formula': 'humanity*0.35 + performance*0.25 + helpfulness*0.40',
        'score_range': '[1.0, 5.0]'
    },
    'ethical_considerations': [
        'Model prioritizes humanity and helpfulness over pure performance',
        'Prompt strategy emphasizes empathy and inclusion',
        'Safety and fairness scores are used as auxiliary features',
        'Model should be used to evaluate, not generate responses'
    ],
    'limitations': [
        'Training data is limited to 15 samples in this demonstration',
        'TF-IDF features depend on language (currently English-only)',
        'Cross-entropy scores should be validated on held-out test data'
    ]
}

with open(f"{CONFIG['output_dir']}model_card.json", 'w') as f:
    json.dump(model_card, f, indent=2)

print("MODEL CARD")
print("=" * 50)
for key, value in model_card.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    elif isinstance(value, list):
        print(f"\n{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"\n{key}: {value}")

In [ ]:
# ============================================
# Cell 16: Final Summary
# ============================================
print("=" * 60)
print("FINAL SUBMISSION - COMPLETE")
print("=" * 60)
print(f"\nModel: Gemma 4 - Good AI for Humanity")
print(f"Architecture: Stacking Ensemble (4 base models)")
print(f"Features: {len(STRUCTURAL_COLS)} structural + {CONFIG['svd_components']} TF-IDF/SVD")
print(f"CV RMSE: {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Training R2: {train_r2:.4f}")
print(f"\nPrompt Strategy: Role-based humanitarian advisor")
print(f"Scoring: Humanity({CONFIG['humanity_weight']}) + Performance({CONFIG['performance_weight']}) + Helpfulness({CONFIG['helpfulness_weight']})")
print(f"\nOutput files:")
print(f"  - submission_template.csv")
print(f"  - final_model_artifacts.pkl")
print(f"  - model_card.json")
print(f"  - final_model_performance.png")

---
## ✅ Final Submission Complete

**Project:** Gemma 4 - Good AI for Humanity, Not Just Performance  
**Status:** Ready for submission  

**Pipeline Summary:**
1. `eda_environment.ipynb` → Domain insights on environment themes
2. `eda_health.ipynb` → Healthcare-specific feature patterns
3. `baseline_model.ipynb` → Structural feature baseline (RMSE benchmark)
4. `tfidf_experiment.ipynb` → Semantic text features (SVD-reduced TF-IDF)
5. `gemma_prompt_test.ipynb` → Optimal prompt engineering strategy
6. `gemma_4_good_final.ipynb` → **Combined ensemble with all improvements**